# Phase 1 — Multimodal Ingestion

**Weeks 1–3 · Goal:** Ingest text, tables, figures, and page images from PDFs into Qdrant with modality-specific embeddings.

## What you will learn

- How PDFs become typed **chunks** (`text`, `table`, `figure`, `page_image`)
- Why each chunk type gets its own **Qdrant collection** and embedding model
- How **Reciprocal Rank Fusion (RRF)** merges results across collections at query time
- The unified entry point: `RAGPipeline` in `shared/pipeline.py`

> **Code:** `phases/phase_01_multimodal_ingestion/` · **Demo CLI:** `phases/phase_01_multimodal_ingestion/demo.py`


In [ ]:
import sys
from pathlib import Path

# Run from notebooks/ — project root is one level up
ROOT = Path.cwd()
if not (ROOT / "shared").exists() and (ROOT.parent / "shared").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT.resolve()}")


## Architecture

```
PDF ──► Parsers ──► DocumentChunks ──► embed_chunks() ──► Qdrant
         │ text/table/figure/page          │ bge-m3 / CLIP / ColPali
         └─────────────────────────────────┴──► 4 collections
```

| Parser | Chunk type | Collection | Embedder |
|--------|------------|------------|----------|
| `PDFTextParser` | `text` | `text_chunks` | bge-m3 (1024-d) |
| `TableParser` | `table` | `table_chunks` | bge-m3 |
| `FigureParser` | `figure` | `figure_chunks` | CLIP (512-d) |
| `PageImageParser` | `page_image` | `page_chunks` | bge-m3 or ColPali (128-d) |


In [ ]:
from shared.models import ChunkType
from phases.phase_01_multimodal_ingestion.stores.qdrant_store import COLLECTION_MAP

for chunk_type, collection in COLLECTION_MAP.items():
    print(f"{chunk_type.value:12} → {collection}")


## Inspect parsers (no Qdrant required)

The sample report PDF is generated by `scripts/generate_sample_report.py`. If it exists, we parse it locally and count chunk types.


In [ ]:
from collections import Counter

from phases.phase_01_multimodal_ingestion.parsers.figure_parser import FigureParser
from phases.phase_01_multimodal_ingestion.parsers.page_image_parser import PageImageParser
from phases.phase_01_multimodal_ingestion.parsers.pdf_text_parser import PDFTextParser
from phases.phase_01_multimodal_ingestion.parsers.table_parser import TableParser

sample = ROOT / "data/raw/sample_report.pdf"
if not sample.exists():
    print("Run: python scripts/generate_sample_report.py")
else:
    parsers = [PDFTextParser(), TableParser(), FigureParser(), PageImageParser()]
    chunks = []
    for p in parsers:
        chunks.extend(p.parse(sample))
    counts = Counter(c.chunk_type.value for c in chunks)
    print(f"Parsed {len(chunks)} chunks from {sample.name}:")
    for kind, n in sorted(counts.items()):
        print(f"  {kind}: {n}")
    print("\nExample chunk:")
    print(chunks[0].model_dump(exclude={"metadata"}))


## Ingest + retrieve (requires Qdrant)

Start Qdrant: `cd docker && docker compose up -d qdrant`

Then ingest and run a retrieval-only query (no LLM needed).


In [ ]:
from shared.models import QueryRequest
from shared.pipeline import PipelineConfig, RAGPipeline

sample = ROOT / "data/raw/sample_report.pdf"
if not sample.exists():
    print("Missing sample PDF — skip ingest demo")
else:
    try:
        cfg = PipelineConfig(text_only=True, use_hybrid=False)
        pipe = RAGPipeline(config=cfg)
        result = pipe.ingest(str(sample))
        print(f"Ingested {result.chunks_upserted} chunks → doc_id={result.doc_id}")

        resp = pipe.query(
            QueryRequest(
                query="What does Figure 3 show about revenue trends?",
                doc_id=result.doc_id,
                top_k=5,
                retrieve_only=True,
            )
        )
        for i, ctx in enumerate(resp.contexts[:3], 1):
            print(f"{i}. [{ctx.chunk_type}] p{ctx.page_number} score={ctx.score:.3f}")
            print(f"   {ctx.content[:120]}...")
    except Exception as exc:
        print(f"Qdrant or models unavailable: {exc}")


## Key concepts

1. **Modality routing** — Questions about charts route to `figure_chunks`; table questions to `table_chunks` via query hints + RRF weights.
2. **Dimension safety** — `page_chunks` must match ingest mode (ColPali 128-d vs bge-m3 1024-d); see `page_index_status()`.
3. **Single pipeline** — Phase 1 demo, eval, and API all call `RAGPipeline`.

**Next:** [02_long_doc_retrieval.ipynb](02_long_doc_retrieval.ipynb) — hybrid search and long-document chunking.
